In [1]:
# just get the data from the json input file and then use langgraph to go through that data and then create a repsonse.

import json
from pprint import pprint

In [2]:
data = []
customer_ids = []
order_ids = []

path = r"D:\projects\AI_SupportOps\CartNest_AI_SupportOps\data\evaluation_tickets_inputs_only.jsonl"
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))
            line = json.loads(line)
            customer_ids.append(line.get("customer_id", None))
            order_ids.append(line.get("order_id", None))

In [3]:
pprint(data[0], indent=4)
pprint(customer_ids[0])
pprint(order_ids[0])

{   'channel': 'email',
    'created_at': '2026-11-18T05:00:00Z',
    'customer_id': 'C0064',
    'message': 'Please tell me when my package CN-00015 should arrive.',
    'order_id': 'CN-00015',
    'status': 'new',
    'subject': 'Delivery Tracking',
    'ticket_id': 'T-EVALUATION-0161'}
'C0064'
'CN-00015'


In [7]:
# Now I am thinking that I'll make an agent that will get this dictionary data. After getting this dictionary data, 
# it will check the customer ID to see if that specific customer ID exists in the list or not, and the same as for order ID. 
# If both of them exist, then the agent will basically output a response or something like that. For now, let me just do this. 

import os

from dotenv import load_dotenv
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek
from langchain.tools import tool

load_dotenv()

True

In [ ]:
model = ChatDeepSeek(model='deepseek-chat')

# we can also create tools to retieve customer id and order id as well.

@tool
def check_customer_order(customer_id: str | None, order_id: str | None) -> str:

    """Use this tool to Check if a customer and their order exist."""

    if customer_id in customer_ids and order_id in order_ids:
        return f"Customer ID {customer_id} and Order ID {order_id} both exist."
    elif customer_id in customer_ids:
        return f"Customer ID {customer_id} exists, but Order ID {order_id} does not. This is ok if the question is not about the order."
    elif order_id in order_ids:
        return f"Order ID {order_id} exists, but Customer ID {customer_id} does not. The customers should be authentic. "
    else:
        return f"Neither Customer ID {customer_id} nor Order ID {order_id} exist. The Customer should be authentic"

In [9]:
agent = create_agent(
    model=model,
    tools=[check_customer_order],
    system_prompt="""
        You are a helpful assistant. Your job is to check whether the customer is authenticated or not. 
        You can use the check customer order tool. You can also check if the order of that specific customer exists or not.
        Your only reason for existence is to check about the support ticket. As a starter, you're checking the customer ID and order ID.
    """
)

In [12]:
message = HumanMessage(
    content=f"""
        customerId: {customer_ids[0]}
        orderId: {order_ids[0]}
        message: {data[0].get("message", None)}
"""
)
response = agent.invoke({"messages": [message]})

In [13]:
pprint(response)

{'messages': [HumanMessage(content='\n        customerId: C0064\n        orderId: CN-00015\n        message: Please tell me when my package CN-00015 should arrive.\n', additional_kwargs={}, response_metadata={}, id='8fef3377-564a-45b7-9bec-78a6f8d077c8'),
              AIMessage(content="I'll check whether this customer and their order exist in the system.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 422, 'total_tokens': 497, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 166}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '5beb7cba-cb87-4dd4-9769-a238db60b19b', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call'},

In [15]:
print(response['messages'][-1].content)

The customer and order both exist in the system:

- **Customer ID:** C0064 — authenticated/verified
- **Order ID:** CN-00015 — found

Regarding the delivery timing question: I'm only able to verify customer authentication and confirm that the order exists. I don't have access to shipment or tracking details, so I can't provide an estimated arrival date for package CN-00015. That inquiry would need to be handled by a team with access to shipping/tracking systems.
